# 02 · BKT Baseline (M1)

Validates the NumPy reimplementation of the paper's pyBKT baseline against the figures reported in Scarlatos, Baker and Lan, *Exploring Knowledge Tracing in Tutor-Student Dialogues using LLMs* (LAK 2025), Table 2, MathDial row for BKT.

**Reported figures (MathDial, BKT):** Acc 60.71, AUC 64.19, F1 56.71. The paper computes metrics on all correctness labels except the first label in each dialogue, which corresponds to `paper_aligned_evaluation` below.

In [1]:
from pathlib import Path
import pandas as pd
from scripts.bkt_model import BKTModel
from scripts.load_data import load_paper_filtered_data

TRAIN_CSV = "data/mathdial_train.csv"
TEST_CSV = "data/mathdial_test.csv"

## Load and filter both splits

`load_paper_filtered_data` applies the paper's preprocessing, the typicality threshold, the failed-annotation and KC-less filters, the final-turn self-correctness override, and the minimum of two tagged turns per dialogue. One documented consequence, the override maps "Yes, but I had to reveal the answer" to NA, so reveal-ending dialogues lose their final turn.

In [2]:
train_df = load_paper_filtered_data(TRAIN_CSV)
test_df = load_paper_filtered_data(TEST_CSV)


def summarize(dataframe, name):
    turn_zero = (
        dataframe["turn"].astype("string").str.strip().str.lower()
        .isin(["solution", "turn 0", "0"])
    )
    real = dataframe[~turn_zero]
    correct = (
        real["correct"].astype("string").str.strip().str.lower().map({"true": 1, "false": 0})
    ).dropna()
    return {
        "split": name,
        "dialogues": dataframe["dialogue_id"].nunique(),
        "tagged turns": len(correct),
        "% correct": round(100 * correct.mean(), 2),
    }


pd.DataFrame([summarize(train_df, "train"), summarize(test_df, "test")])

,split,dialogues,tagged turns,% correct
0,train,2050,10448,49.82
1,test,515,2500,45.96


## Fit and evaluate

One EM fit per KC at seed 221, matching `BKT(seed=221, num_fits=1)`. `standard_evaluation` scores every retained tagged test turn, `paper_aligned_evaluation` additionally drops each dialogue's first prediction, which is the paper's protocol.

In [3]:
model = BKTModel(train_df, test_df, seed=221)
metrics = model.run()

pd.DataFrame(metrics).T.mul(100).round(2)

,Accuracy,AUC,F1
standard_evaluation,59.96,63.05,55.77
paper_aligned_evaluation,60.65,64.28,55.60


## Comparison against the paper's reported figures

In [5]:
PAPER = {"Accuracy": 60.71, "AUC": 64.19, "F1": 56.71}  # Table 2, MathDial, BKT row

ours = {k: 100 * v for k, v in metrics["paper_aligned_evaluation"].items()}
comparison = pd.DataFrame(
    {
        "reported": PAPER,
        "reproduced": {k: round(ours[k], 2) for k in PAPER},
        "delta": {k: round(ours[k] - PAPER[k], 2) for k in PAPER},
    }
)
comparison

,reported,reproduced,delta
Accuracy,60.71,60.65,-0.06
AUC,64.19,64.28,0.09
F1,56.71,55.60,-1.11


## Result and notes

**M1 is frozen as the baseline of record** at Accuracy 60.65, AUC 64.28, F1 55.60 on the paper-aligned protocol, over the filtered population of 2,050 train and 515 test dialogues (10,448 and 2,500 tagged turns).

- Accuracy and AUC reproduce the paper to within 0.1 points, so the fitted parameters and probability trajectories are essentially pyBKT's.
- The 1.1-point F1 residual traces to pyBKT 1.4.x's E-step zero-likelihood guard, `np.where(sl == 0, 1, sl)` in `fit/EM_fit.py`, verified at source. It moves roughly forty near-threshold predictions across 0.5 without affecting the ranking, which is why AUC is unaffected. Matching it would mean reproducing implementation quirks rather than the statistical model.
- An attempt to run installed pyBKT directly on the same data returned degraded figures (58.44, 61.36, 51.61) with invalid-divide warnings in its E-step on the modern NumPy and scikit-learn stack. Recorded in the audit log as an environment-degraded anchor, not pursued further.
- All downstream comparisons in this project run against this implementation's figures, with the paper's row quoted as the external anchor.